In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from typing import Annotated
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
load_dotenv()


In [ ]:
# Step 1: Define State
# MessagesState already includes messages with add_messages

# Step 2: Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini")

In [2]:
# Step 3: Create Agent Node
def agent(state: MessagesState):
    """
    Call the LLM with full conversation history.
    Messages are automatically accumulated in state.
    """
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

In [3]:
# Step 4: Build Graph
def create_stm_agent():
    """Create STM agent with InMemorySaver"""
    builder = StateGraph(MessagesState)
    builder.add_node("agent", agent)
    builder.add_edge(START, "agent")
    builder.add_edge("agent", END)
    
    # Attach checkpointer (saves state to memory)
    checkpointer = InMemorySaver()
    graph = builder.compile(checkpointer=checkpointer)
    return graph

In [4]:
# Step 5: Run Conversations
def main():
    graph = create_stm_agent()
    
    # Thread 1: User Alice
    config1 = {"configurable": {"thread_id": "thread-1"}}
    
    # Turn 1
    print("=== Thread 1: Alice ===")
    result = graph.invoke(
        {"messages": [HumanMessage(content="Hi! My name is Alice")]},
        config1
    )
    print(f"Bot: {result['messages'][-1].content}\n")
    
    # Turn 2: State is automatically loaded from thread-1
    result = graph.invoke(
        {"messages": [HumanMessage(content="What's my name?")]},
        config1
    )
    print(f"Bot: {result['messages'][-1].content}\n")
    # Bot remembers "Alice" from Turn 1!
    
    # Thread 2: User Bob (completely separate)
    config2 = {"configurable": {"thread_id": "thread-2"}}
    
    print("\n=== Thread 2: Bob ===")
    result = graph.invoke(
        {"messages": [HumanMessage(content="Hi! My name is Bob")]},
        config2
    )
    print(f"Bot: {result['messages'][-1].content}\n")
    
    # Turn 2 in thread 2
    result = graph.invoke(
        {"messages": [HumanMessage(content="What's my name?")]},
        config2
    )
    print(f"Bot: {result['messages'][-1].content}\n")
    # Bot correctly responds with "Bob" (different thread)
    
    # Verify: Both threads maintained separate states
    print("=== Back to Thread 1 ===")
    result = graph.invoke(
        {"messages": [HumanMessage(content="Remind me my name")]},
        config1
    )
    print(f"Bot: {result['messages'][-1].content}")
    # Still remembers Alice!
    

In [5]:
if __name__ == "__main__":
    main()

=== Thread 1: Alice ===
Bot: Hi Alice! How can I assist you today?

Bot: Your name is Alice. How can I help you today, Alice?


=== Thread 2: Bob ===
Bot: Hi, Bob! How can I assist you today?

Bot: Your name is Bob. How can I help you today, Bob?

=== Back to Thread 1 ===
Bot: Your name is Alice. Let me know if there's anything else you need!
